In [2]:
import pandas as pd
import numpy as np
import pickle
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression , Lasso , Ridge
from sklearn.metrics import root_mean_squared_error

In [12]:
def read_dataframe(file_name):
    if file_name.endswith('.parquet'):
        df = pd.read_parquet(file_name)
    elif file_name.endswith('.csv'):
        # FIXED: Added 'pd.' before read_csv
        df = pd.read_csv(file_name) 
        
    df['duration'] = df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
    # FIXED: Convert the duration to minutes
    df['duration'] = df['duration'].apply(lambda td: td.total_seconds() / 60)
    
    # FIXED: Filter out the outliers (typically trips between 1 and 60 minutes)
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df


In [13]:
df_train = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-02.parquet')
df_val = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet')

In [14]:
len(df_train) , len(df_val)

(3258717, 3571924)

In [15]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [16]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [17]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [18]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

6.439287708808353

In [19]:
import os

os.makedirs('models', exist_ok=True)
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [20]:
lr = Lasso(0.01)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

10.324181689096193

In [ ]:
df['duration'].std() #standard deviation of the trips duration

25.217064291507768

In [25]:
df[(df['duration']>=1) & (df['duration']<=60)].shape[0] / df.shape[0] #What fraction of the records left after you dropped the outliers?

0.9584839520145794

In [30]:
df.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee', 'duration'],
      dtype='object')

In [34]:
categorical = ['PULocationID','DOLocationID']
df[categorical] = df[categorical].astype(str)
X = df[categorical].to_dict(orient ='records')
d = DictVectorizer()
X = d.fit_transform(X)
X.shape[1]

520

In [35]:
lr = LinearRegression()
lr.fit(X,df['duration'].values)
y_pred = lr.predict(X)
root_mean_squared_error(df['duration'].values , y_pred)

24.01534108632286